# Peak Clean — Exact-0.01 BBO Taker

Thesis: when a BBO level shows a size of **exactly 0.01 BTC**, that exposed order is about
to get run over — the market moves against it. So we enter in the direction that runs it
over, **by taking that very order**:

| Signal | Read | Action |
|---|---|---|
| exactly 0.01 at the **best ask** | price is about to rise through it | **BUY** — lift that ask |
| exactly 0.01 at the **best bid** | price is about to fall through it | **SELL** — hit that bid |

## Speed & rate-limit design (100 req / 60s budget)
- **Event-driven**: wakes on every TrueMarkets DEPTH websocket message (same feed class as
  `maker_clean`); the signal → order path is in-memory checks plus **one POST** — no REST
  reads sit between detection and execution.
- **Fill usually confirmed from the create response** (`status == 'complete'`), so the common
  case costs **1 request per trade**. If the order comes back live instead, it is polled at
  most every `FILL_CHECK_SECS` and cancelled after `TAKE_TIMEOUT_SECS` — worst case ~5
  requests per attempt, hard-bounded.
- Balances are reconciled every `RECONCILE_SECS` (3 req/min) instead of per loop; position
  is tracked locally from fills in between.
- Entry frequency is capped three ways: a per-entry cooldown, **one-shot arming per
  exposure** (a level only fires once until its price changes — no re-firing on every book
  tick while the 0.01 sits there), and a rate-limit reserve so a cancel is always affordable.

## Risk containment
- **Inventory band is hard-capped at ±0.001 BTC** with a clip of exactly 0.001 per trade —
  the position can only ever be −0.001, 0, or +0.001. A signal that would breach the band is
  skipped; a signal in the flattening direction is preferred when both sides fire at once.
- **No market orders.** Entry is a marketable **limit at the exposed order's exact price** —
  it takes instantly if the 0.01 is still there and by construction can never fill at a worse
  price than the one we saw. If it misses, it is cancelled, never chased.
- `place_taker_limit()` is the only order path and re-validates at the last instant: book
  fresh, not crossed, price still the touch, **trigger size still displayed**, tick-aligned,
  exact 0.001 size.
- Gates carried over from `maker_clean`: book freshness, crossed-book, TM-vs-Coinbase mid
  sanity bound, $5 session loss limit, cancel-all on every exit path.

## Caveats
- The DEPTH feed aggregates by price level: "exactly 0.01" can be several orders summing to
  0.01. Indistinguishable from a single 0.01 order at this feed granularity.
- There is no standalone exit — the position flattens/reverses only on an opposite signal
  (or manual stop). With a ±0.001 band the worst carry is 0.001 BTC.
- A partial fill on a cancelled taker order is booked as all-or-nothing locally; the 20s
  balance reconcile trues up inventory (and fees) authoritatively.

In [ ]:
import sys
import os
import asyncio
import math
import time
import json
from pathlib import Path

ROOT_DIR = Path(os.getcwd()).parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import aiohttp
import websockets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from dotenv import load_dotenv

from lib.execution import ExecutionClient
from lib.coinbase_feed import CoinbaseBookA

load_dotenv(ROOT_DIR / 'keys' / '.env')

TM_REST_URL = os.getenv('BASE_REST_URL', 'https://api.truemarkets.co')
TM_KEY_FILE = str(ROOT_DIR / 'keys' / 'truemarkets-api-key-edd1691b.json')
CB_WS_URL   = os.getenv('COINBASE_WS_URL', 'wss://ws-feed.exchange.coinbase.com')
CB_PRODUCT  = 'BTC-USD'
BASE_ASSET  = 'BTC'
QUOTE_ASSET = 'USDC'


In [ ]:
class TrueMarketsBook:
    _WS_URL = 'wss://api.truex.co/api/v1'
    _HEADERS = {
        'Origin': 'https://truemarkets.co',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
    }

    def __init__(self, symbol='BTC-PYUSD'):
        self.symbol    = symbol
        self._bids     = {}
        self._asks     = {}
        self._ready    = False
        self._last_msg = None            # wall-clock time of last websocket message
        self.updated   = asyncio.Event() # set on every book change — lets the strategy react instantly

    @property
    def best_bid(self):      return max(self._bids) if self._bids else None
    @property
    def best_ask(self):      return min(self._asks) if self._asks else None
    @property
    def best_bid_size(self):
        b = self.best_bid; return self._bids[b] if b is not None else None
    @property
    def best_ask_size(self):
        a = self.best_ask; return self._asks[a] if a is not None else None
    @property
    def mid(self):
        b, a = self.best_bid, self.best_ask
        return (b + a) / 2.0 if (b is not None and a is not None) else None
    @property
    def is_ready(self): return self._ready
    @property
    def age(self):
        """Seconds since the last websocket message, or None if never connected."""
        return (time.time() - self._last_msg) if self._last_msg else None

    def _handle_snapshot(self, data):
        self._bids  = {float(b['price']): float(b['qty']) for b in data.get('bids', []) if float(b['qty']) > 0}
        self._asks  = {float(a['price']): float(a['qty']) for a in data.get('asks', []) if float(a['qty']) > 0}
        self._ready = True
        self.updated.set()

    def _handle_update(self, data):
        for b in data.get('bids', []):
            p, q = float(b['price']), float(b['qty'])
            if q == 0: self._bids.pop(p, None)
            else:      self._bids[p] = q
        for a in data.get('asks', []):
            p, q = float(a['price']), float(a['qty'])
            if q == 0: self._asks.pop(p, None)
            else:      self._asks[p] = q
        self.updated.set()

    async def run(self):
        backoff = 1
        while True:
            try:
                async with websockets.connect(self._WS_URL, additional_headers=self._HEADERS) as ws:
                    backoff = 1
                    self._ready = False
                    await ws.send(json.dumps({
                        'type': 'SUBSCRIBE_NO_AUTH',
                        'item_names': [self.symbol],
                        'channels': ['DEPTH'],
                        'timestamp': str(int(time.time())),
                    }))
                    async for raw in ws:
                        self._last_msg = time.time()
                        try: msg = json.loads(raw)
                        except Exception: continue
                        t = msg.get('update')
                        d = msg.get('data', {})
                        if   t == 'SNAPSHOT': self._handle_snapshot(d)
                        elif t == 'UPDATE':   self._handle_update(d)
            except websockets.ConnectionClosed:
                pass
            except Exception as e:
                print(f'TrueMarketsBook error: {e}')
            self._ready = False
            self.updated.set()   # wake the strategy so it notices the outage immediately
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)

### Parameters

In [ ]:
TAKE_SIZE_BTC      = 0.001   # EVERY take is exactly this size
TRIGGER_SIZE_BTC   = 0.01    # fire only when a BBO level shows EXACTLY this size
SIZE_EPS           = 1e-9    # float tolerance for the exact-size comparison
TICK_SIZE          = 0.1
PRICE_DECIMALS     = 1
IDLE_TICK_SECS     = 1.0     # heartbeat when the websocket is quiet (housekeeping only)

# ── risk limits ──────────────────────────────────────────────────────────────
MAX_POSITION_BTC   = 0.001   # hard inventory band: never net long/short more than this
DAILY_LOSS_LIMIT   = 5.0     # session kill switch ($)

# ── entry pacing / rate-limit protection ─────────────────────────────────────
TAKE_COOLDOWN_SECS = 2.0     # min gap between entry attempts
TAKE_TIMEOUT_SECS  = 3.0     # unfilled taker limit is cancelled after this — never chase
FILL_CHECK_SECS    = 0.8     # min gap between status polls on an in-flight order
RL_RESERVE         = 10      # keep this many requests in reserve so a cancel is always possible
RECONCILE_SECS     = 20.0    # balance/inventory reconciliation cadence (saves rate limit)

# ── safety gates ─────────────────────────────────────────────────────────────
MAX_BOOK_AGE_SECS  = 5.0     # taking on a stale book is blind — tighter than the maker's 10s
MAX_CB_DEVIATION   = 0.01    # TM mid vs Coinbase mid sanity bound (1%): no entries

PLOT_EVERY_SECS    = 5.0
STATUS_EVERY_SECS  = 2.0
TM_BOOK_SYMBOL     = 'BTC-PYUSD'

# order statuses that mean "still live on the exchange"
LIVE_STATUSES = {'pending', 'active', 'open', 'new', 'partially_filled'}

### Helpers

In [ ]:
size_str = f'{TAKE_SIZE_BTC:.8f}'.rstrip('0').rstrip('.')
assert size_str == '0.001', f'take size must be exactly 0.001, got {size_str}'

def fmt_px(p): return f'{p:.{PRICE_DECIMALS}f}'

def is_trigger(sz):
    """True when a displayed size is EXACTLY the 0.01 trigger."""
    return sz is not None and abs(sz - TRIGGER_SIZE_BTC) < SIZE_EPS


async def place_taker_limit(bot, session, tm_book, side, px):
    """The ONLY order-placement path. A marketable limit at the exposed order's
    exact price: it takes instantly if the 0.01 is still displayed, and by
    construction can never fill at a worse price than the one we saw. Everything
    is re-validated against the live book at the last possible instant — if the
    trigger vanished in flight, we refuse rather than rest or chase.

    Returns (order, refusal_reason) — exactly one of the two is set."""
    if abs(TAKE_SIZE_BTC - 0.001) > 1e-12:
        return None, 'size drifted from 0.001'
    if abs(px - round(px / TICK_SIZE) * TICK_SIZE) > 1e-6:
        return None, f'price {px} not tick-aligned'
    px_str = fmt_px(px)
    if abs(float(px_str) - px) > 1e-6:
        return None, f'price string {px_str} != {px}'
    if not tm_book.is_ready or tm_book.age is None or tm_book.age > MAX_BOOK_AGE_SECS:
        return None, 'book stale at send time'
    lb, la = tm_book.best_bid, tm_book.best_ask
    if lb is None or la is None or lb >= la:
        return None, 'book empty/crossed at send time'
    if side == 'buy':
        if abs(px - la) > 1e-9:
            return None, f'ask moved to {la:.1f} != {px_str}'
        if not is_trigger(tm_book.best_ask_size):
            return None, 'trigger size gone from ask'
    elif side == 'sell':
        if abs(px - lb) > 1e-9:
            return None, f'bid moved to {lb:.1f} != {px_str}'
        if not is_trigger(tm_book.best_bid_size):
            return None, 'trigger size gone from bid'
    else:
        return None, f'bad side {side!r}'
    order = await bot.place_order(
        session, BASE_ASSET, QUOTE_ASSET,
        side, size_str, 'base', 'limit', px_str
    )
    return order, None


async def get_full_balances(bot, session):
    data = await bot._get(session, '/v1/conductor/balances')
    total_btc = avail_btc = total_usd = avail_usd = 0.0
    if not data:
        return None, None, None, None
    for b in data.get('data', []):
        sym   = b.get('symbol', '')
        avail = float(b.get('available', 0) or 0)
        held  = float(b.get('held',      0) or 0)
        if sym == BASE_ASSET:
            total_btc += avail + held
            avail_btc += avail
        elif sym in (QUOTE_ASSET, 'PYUSD'):
            total_usd += avail + held
            avail_usd += avail
    return total_btc, total_usd, avail_btc, avail_usd

# ── chart series ─────────────────────────────────────────────────────────────
time_series = []
pnl_series  = []
vol_series  = []
inv_series  = []
trades      = []   # (elapsed_secs, side, px)

def _cumsum(vals):
    out, s = [], 0.0
    for v in vals:
        s += v; out.append(s)
    return out

def update_plot(tm_book=None, last_fill_px=None, last_fill_side=None):
    clear_output(wait=True)
    fig, axes = plt.subplots(4, 1, figsize=(12, 16))
    ax1, ax2, ax3, ax4 = axes

    ax1.plot(time_series, pnl_series, color='green', label='Session PnL ($)')
    ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax1.axhline(-DAILY_LOSS_LIMIT, color='red', linestyle='--', linewidth=1,
                label=f'Loss limit -${DAILY_LOSS_LIMIT}')
    ax1.fill_between(time_series, pnl_series, 0,
                     where=[p >= 0 for p in pnl_series], alpha=0.15, color='green')
    ax1.fill_between(time_series, pnl_series, 0,
                     where=[p <  0 for p in pnl_series], alpha=0.15, color='red')
    ax1.set_ylabel('PnL ($)'); ax1.set_title('Session PnL')
    ax1.legend(loc='upper left'); ax1.grid(True, alpha=0.3)

    ax2.plot(time_series, vol_series, color='blue', label='Taker volume ($)')
    ax2.fill_between(time_series, vol_series, alpha=0.12, color='blue')
    ax2.set_ylabel('Volume ($)'); ax2.set_title('Cumulative Taker Volume')
    ax2.legend(loc='upper left'); ax2.grid(True, alpha=0.3)

    ax3.plot(time_series, inv_series, color='orange', label='Net inventory (BTC)')
    ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.axhline( MAX_POSITION_BTC, color='red', linestyle=':', alpha=0.6, label='Inventory band')
    ax3.axhline(-MAX_POSITION_BTC, color='red', linestyle=':', alpha=0.6)
    buys  = [(t, TAKE_SIZE_BTC)  for t, s, _ in trades if s == 'buy']
    sells = [(t, -TAKE_SIZE_BTC) for t, s, _ in trades if s == 'sell']
    if buys:
        ax3.scatter([t for t, _ in buys],  [y for _, y in buys],  marker='^',
                    color='green', s=60, zorder=5, label='Buy takes')
    if sells:
        ax3.scatter([t for t, _ in sells], [y for _, y in sells], marker='v',
                    color='red',   s=60, zorder=5, label='Sell takes')
    ax3.set_ylabel('BTC'); ax3.set_title(f'Net Inventory — {len(trades)} takes')
    ax3.legend(loc='upper left'); ax3.grid(True, alpha=0.3)

    if tm_book is not None and tm_book.is_ready and tm_book._bids and tm_book._asks:
        sorted_bids = sorted(tm_book._bids.items(), reverse=True)
        sorted_asks = sorted(tm_book._asks.items())
        bid_px  = [p for p, _ in sorted_bids]
        bid_cum = _cumsum([q for _, q in sorted_bids])
        ask_px  = [p for p, _ in sorted_asks]
        ask_cum = _cumsum([q for _, q in sorted_asks])
        ax4.fill_between(bid_px, bid_cum, step='post', color='green', alpha=0.35, label='Bids')
        ax4.plot(bid_px, bid_cum, color='lime',   drawstyle='steps-post', linewidth=1)
        ax4.fill_between(ask_px, ask_cum, step='pre',  color='red',   alpha=0.35, label='Asks')
        ax4.plot(ask_px, ask_cum, color='salmon', drawstyle='steps-pre',  linewidth=1)
        if last_fill_px is not None:
            c = 'lime' if last_fill_side == 'buy' else 'salmon'
            ax4.axvline(last_fill_px, color=c, linestyle='--', linewidth=1.5,
                        label=f'Last take {last_fill_side} ${last_fill_px:.1f}')
        mid = (bid_px[0] + ask_px[0]) / 2
        ax4.set_xlim(mid - 500, mid + 500)
    else:
        ax4.text(0.5, 0.5, 'Waiting for order book...', ha='center', va='center',
                 transform=ax4.transAxes)
    ax4.set_xlabel('Price ($)'); ax4.set_ylabel('Cumulative BTC')
    ax4.set_title('Live TrueMarkets Order Book Depth')
    ax4.legend(loc='upper right'); ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    display(fig)
    plt.close(fig)

### Strategy Loop

Evaluated on every websocket event; the signal check is the first thing after the gates so
detection → POST has nothing slow in between.

| Book shows | Meaning | Action |
|---|---|---|
| exactly 0.01 at best ask, side armed | exposed seller about to be run over | **BUY** — marketable limit at that ask |
| exactly 0.01 at best bid, side armed | exposed buyer about to be run over | **SELL** — marketable limit at that bid |
| both at once | ambiguous | take the side that reduces inventory |
| trigger fired at this price already | one-shot per exposure | wait until the BBO price changes (re-arm) |
| order in flight | at most one order ever | poll status (throttled), cancel after timeout |

A signal is skipped (but still disarms that exposure) when: gates are up, cooldown active,
the trade would breach the ±0.001 band, or balances can't cover it.

In [ ]:
async def run_strategy():
    for s in [time_series, pnl_series, vol_series, inv_series]:
        s.clear()
    trades.clear()

    bot     = ExecutionClient(key_file=TM_KEY_FILE, base_url=TM_REST_URL)
    book_a  = CoinbaseBookA(ws_url=CB_WS_URL, product_id=CB_PRODUCT)
    tm_book = TrueMarketsBook(symbol=TM_BOOK_SYMBOL)

    feed_cb = asyncio.create_task(book_a.run())
    feed_tm = asyncio.create_task(tm_book.run())

    start_time = time.time()

    pending           = None   # the single in-flight taker order (dict) or None
    last_fired_bid_px = None   # one-shot arming: BBO price we already fired on
    last_fired_ask_px = None
    last_take_at      = 0.0

    initial_btc = initial_usd = None
    inv = cash = taker_volume = 0.0
    avail_btc = avail_usd = 0.0
    last_fill_px = last_fill_side = None

    last_reconcile_at = last_plot_at = last_print_at = 0.0

    def record_fill(side, px):
        nonlocal inv, cash, taker_volume, avail_btc, avail_usd, last_fill_px, last_fill_side
        signed = TAKE_SIZE_BTC if side == 'buy' else -TAKE_SIZE_BTC
        inv  += signed
        cash -= signed * px
        if side == 'buy':
            avail_usd -= TAKE_SIZE_BTC * px
            avail_btc += TAKE_SIZE_BTC
        else:
            avail_btc -= TAKE_SIZE_BTC
            avail_usd += TAKE_SIZE_BTC * px
        taker_volume += TAKE_SIZE_BTC * px
        last_fill_px, last_fill_side = px, side
        trades.append((time.time() - start_time, side, px))
        print(f'  ✓ TAKE {side.upper()} {size_str} @ ${px:.1f}  inv {inv:+.5f}')

    try:
        async with aiohttp.ClientSession() as session:

            async def resolve_pending(try_cancel):
                """Check the in-flight taker order (cancelling first when asked).
                Books the fill if it completed. True once the order is resolved.
                A cancel can race a fill — status after the cancel is what counts."""
                nonlocal pending
                if not pending:
                    return True
                if try_cancel:
                    try:
                        await bot.cancel_order(session, pending['oid'])
                    except Exception:
                        pass
                st = await bot.get_order_status(session, pending['oid'])
                if st == 'complete':
                    record_fill(pending['side'], pending['px'])
                    pending = None
                    return True
                if st is not None and st not in LIVE_STATUSES:
                    print(f'  ! taker order ended {st} without a booked fill')
                    pending = None
                    return True
                return False

            await bot.authenticate(session)
            print('Authenticated. Cancelling any pre-existing orders...')
            await bot.cancel_all(session)

            res = await get_full_balances(bot, session)
            if res[0] is None:
                raise RuntimeError('Failed to fetch initial balances')
            initial_btc, initial_usd, avail_btc, avail_usd = res
            print(f'Starting balances: {initial_btc:.6f} BTC  |  ${initial_usd:.2f} USDC+PYUSD')
            print('Waiting for TrueMarkets book snapshot...')

            while not tm_book.is_ready:
                await asyncio.sleep(0.2)
            print(f'TM book ready: {tm_book.best_bid:.1f} / {tm_book.best_ask:.1f}')
            print(f'Hunting for BBO levels of exactly {TRIGGER_SIZE_BTC} BTC — interrupt kernel to stop.\n')

            while True:
                # ── 0. Wake on the next book event (or heartbeat) ─────────────
                try:
                    await asyncio.wait_for(tm_book.updated.wait(), timeout=IDLE_TICK_SECS)
                except asyncio.TimeoutError:
                    pass
                tm_book.updated.clear()
                now = time.time()

                # ── 1. Book freshness — never take on a stale/dead book ───────
                age   = tm_book.age
                fresh = tm_book.is_ready and age is not None and age <= MAX_BOOK_AGE_SECS
                if not fresh:
                    if pending:
                        print(f"  {time.strftime('%H:%M:%S')}  feed stale with order in flight — pulling it")
                        await resolve_pending(try_cancel=True)
                    await asyncio.sleep(IDLE_TICK_SECS)
                    continue

                best_bid = tm_book.best_bid
                best_ask = tm_book.best_ask
                if best_bid is None or best_ask is None:
                    await asyncio.sleep(IDLE_TICK_SECS)
                    continue
                bid_sz = tm_book.best_bid_size or 0.0
                ask_sz = tm_book.best_ask_size or 0.0

                # ── 2. Entry gates (all in-memory — nothing slow before the POST)
                cb_mid = book_a.mid
                tm_mid = tm_book.mid
                rl     = bot._rl_remaining

                crossed   = best_bid >= best_ask
                deviated  = bool(cb_mid and tm_mid and abs(tm_mid - cb_mid) / cb_mid > MAX_CB_DEVIATION)
                budget_ok = (rl is None) or (rl > RL_RESERVE)

                quote_gate = None
                if crossed:
                    quote_gate = f'book crossed {best_bid:.1f}/{best_ask:.1f}'
                elif deviated:
                    quote_gate = f'TM mid {tm_mid:.1f} vs CB mid {cb_mid:.1f} > {MAX_CB_DEVIATION:.1%}'
                elif not budget_ok:
                    quote_gate = f'rate budget {rl} <= reserve {RL_RESERVE}'

                mark = cb_mid or tm_mid

                # ── 3. FAST PATH: exact-0.01 BBO detection → single POST ──────
                # re-arm a side as soon as its BBO price moves off the fired level
                if last_fired_ask_px is not None and abs(best_ask - last_fired_ask_px) > 1e-9:
                    last_fired_ask_px = None
                if last_fired_bid_px is not None and abs(best_bid - last_fired_bid_px) > 1e-9:
                    last_fired_bid_px = None

                buy_sig  = is_trigger(ask_sz) and last_fired_ask_px is None
                sell_sig = is_trigger(bid_sz) and last_fired_bid_px is None

                side = None
                if buy_sig and sell_sig:
                    side = 'sell' if inv > 0 else 'buy'   # both exposed: reduce inventory
                elif buy_sig:
                    side = 'buy'
                elif sell_sig:
                    side = 'sell'

                action = 'watching'
                if side and pending is None:
                    px = best_ask if side == 'buy' else best_bid
                    # one shot per exposure — this level won't fire again until its price changes
                    if side == 'buy':
                        last_fired_ask_px = best_ask
                    else:
                        last_fired_bid_px = best_bid

                    if quote_gate:
                        action = f'{side} signal @ {px:.1f} skipped ({quote_gate})'
                    elif now - last_take_at < TAKE_COOLDOWN_SECS:
                        action = f'{side} signal @ {px:.1f} skipped (cooldown)'
                    elif side == 'buy' and inv + TAKE_SIZE_BTC > MAX_POSITION_BTC + 1e-12:
                        action = f'buy signal @ {px:.1f} skipped (max long {inv:+.5f})'
                    elif side == 'sell' and inv - TAKE_SIZE_BTC < -MAX_POSITION_BTC - 1e-12:
                        action = f'sell signal @ {px:.1f} skipped (max short {inv:+.5f})'
                    elif side == 'buy' and avail_usd < TAKE_SIZE_BTC * px * 1.01:
                        action = f'buy signal @ {px:.1f} skipped (no USDC)'
                    elif side == 'sell' and avail_btc < TAKE_SIZE_BTC:
                        action = f'sell signal @ {px:.1f} skipped (no BTC)'
                    else:
                        order, refusal = await place_taker_limit(bot, session, tm_book, side, px)
                        last_take_at = now
                        if order and order.get('order_id'):
                            st = order.get('status')
                            if st == 'complete':
                                record_fill(side, px)
                                action = f'TAKE {side.upper()} filled on create @ {px:.1f}'
                            else:
                                pending = {'oid': order['order_id'], 'side': side, 'px': px,
                                           'placed_at': now, 'checked_at': now}
                                action = f'TAKE {side.upper()} {size_str} @ {px:.1f} sent ({st})'
                        elif refusal:
                            action = f'refused ({refusal})'
                        else:
                            action = f'TAKE {side.upper()} FAILED'

                # ── 4. In-flight order management (throttled polls, hard timeout)
                if pending and now - pending['checked_at'] >= FILL_CHECK_SECS:
                    pending['checked_at'] = now
                    timed_out = now - pending['placed_at'] >= TAKE_TIMEOUT_SECS
                    if timed_out:
                        print(f'  ! taker order unfilled after {TAKE_TIMEOUT_SECS:.0f}s — cancelling, not chasing')
                    await resolve_pending(try_cancel=timed_out)

                # ── 5. Balance reconciliation (authoritative inventory) ───────
                if now - last_reconcile_at >= RECONCILE_SECS:
                    last_reconcile_at = now
                    res = await get_full_balances(bot, session)
                    if res[0] is not None:
                        total_btc, total_usd, avail_btc, avail_usd = res
                        new_inv = total_btc - initial_btc
                        if abs(new_inv - inv) > 1e-9:
                            print(f'  ~ reconcile: inv {inv:+.6f} -> {new_inv:+.6f}')
                        inv  = new_inv
                        cash = total_usd - initial_usd

                # ── 6. Chart and status (throttled — actions are not) ─────────
                total_pnl = cash + inv * (mark or 0)
                elapsed   = time.time() - start_time

                time_series.append(elapsed)
                pnl_series.append(total_pnl)
                vol_series.append(taker_volume)
                inv_series.append(inv)

                if now - last_plot_at >= PLOT_EVERY_SECS:
                    last_plot_at = now
                    update_plot(tm_book, last_fill_px, last_fill_side)

                acted = action != 'watching'
                if acted or now - last_print_at >= STATUS_EVERY_SECS:
                    last_print_at = now
                    print(
                        f"  {time.strftime('%H:%M:%S')}  "
                        f"book {best_bid:.1f}[{bid_sz:.4f}]/{best_ask:.1f}[{ask_sz:.4f}]  "
                        f"inv {inv:+.5f}  pnl ${total_pnl:+.4f}  "
                        f"vol ${taker_volume:.2f}  takes {len(trades)}  "
                        f"rl {rl if rl is not None else '?'}/100"
                    )
                    if acted:
                        print(f'    {action}')
                    elif pending:
                        print(f"    awaiting fill: {pending['side']} {size_str} @ {pending['px']:.1f}")

                # ── 7. Kill switch ────────────────────────────────────────────
                if total_pnl < -DAILY_LOSS_LIMIT:
                    print(f'\nLoss limit hit (${total_pnl:.4f}). Cancelling all orders.')
                    await bot.cancel_all(session)
                    break

    except (asyncio.CancelledError, KeyboardInterrupt):
        print('\nInterrupted.')
    finally:
        feed_cb.cancel()
        feed_tm.cancel()
        # never leave orders on the book, no matter how we exited
        try:
            async with aiohttp.ClientSession() as cleanup_session:
                await bot.cancel_all(cleanup_session)
            print('All orders cancelled on exit.')
        except BaseException as e:
            print(f'!! cancel_all on exit failed ({e!r}) — CHECK OPEN ORDERS MANUALLY')
        print('Stopped.')

In [ ]:
# Interrupt kernel (stop button) to exit cleanly — all orders are cancelled on the way out
await run_strategy()